# Phase 1: Dataset Inspection and Understanding
## Daily Electricity Consumption Prediction in Smart Buildings (BDG2 Dataset)

- **Project Title:** Daily Electricity Consumption Prediction in Smart Buildings Using Machine Learning
- **Dataset:** Building Data Genome Project 2 (BDG2)
- **Academic Context:** 4-Member Team Academic ML Project (3rd Year Information Science & Engineering)
- **Official Source:** Zenodo Record [3887306](https://zenodo.org/records/3887306) (Miller et al., 2020)
- **Phase Objective:** Thoroughly inspect the raw BDG2 dataset, discover folder hierarchy, verify file formats, column schemas, measurement units, data types, temporal resolution, missingness, and cross-dataset linkage without making assumptions or modifying raw data.


### 1. Environment Setup and Project Imports
We import standard data science libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and our reusable data loader utility module `src/data_loader.py`.


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path so we can import src modules
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import get_raw_data_dir, load_metadata, load_weather, load_electricity, get_dataset_inventory

print(f"Project Root: {project_root}")
raw_data_dir = get_raw_data_dir(project_root)
print(f"Discovered BDG2 Raw Data Directory: {raw_data_dir}")


Project Root: C:\Users\ADMIN\Desktop\smart-building-energy-prediction
Discovered BDG2 Raw Data Directory: C:\Users\ADMIN\Desktop\smart-building-energy-prediction\data\raw\buds-lab-building-data-genome-project-2-3d0cbaf\data


### 2. Dataset Inventory and File Organization
Let us scan all files inside the BDG2 dataset directory to inspect their categories, relative paths, file formats, and storage sizes.


In [2]:
inventory_df = get_dataset_inventory(raw_data_dir)
print(f"Total BDG2 files discovered: {len(inventory_df)}")
display(inventory_df)


Total BDG2 files discovered: 19


,Category,Relative Path,Size (MB),Extension
0,metadata,metadata\metadata.csv,0.26,.csv
9,meters,meters\kaggle\kaggle.csv,855.27,.csv
2,meters,meters\cleaned\electricity_cleaned.csv,169.25,.csv
11,meters,meters\raw\electricity.csv,166.17,.csv
10,meters,meters\raw\chilledwater.csv,75.19,.csv
1,meters,meters\cleaned\chilledwater_cleaned.csv,73.44,.csv
16,meters,meters\raw\steam.csv,44.58,.csv
7,meters,meters\cleaned\steam_cleaned.csv,43.32,.csv
13,meters,meters\raw\hotwater.csv,23.79,.csv
4,meters,meters\cleaned\hotwater_cleaned.csv,23.23,.csv


### 3. Inspecting Building Metadata (`metadata.csv`)
Let us examine the metadata file containing static physical and operational attributes for all buildings in the dataset.


In [3]:
meta_df = load_metadata(raw_data_dir)
print(f"Metadata Dimensions: {meta_df.shape[0]} buildings (rows), {meta_df.shape[1]} attributes (columns)")
display(meta_df.head(5))


Metadata Dimensions: 1636 buildings (rows), 32 attributes (columns)


,building_id,site_id,building_id_kaggle,site_id_kaggle,primaryspaceusage,sub_primaryspaceusage,sqm,sqft,lat,lng,...,yearbuilt,date_opened,numberoffloors,occupants,energystarscore,eui,site_eui,source_eui,leed_level,rating
0,Panther_lodging_Dean,Panther,NaN,0.0,Lodging/residential,Residence Hall,508.8,5477.0,28.517689,-81.379039,...,1989.0,NaN,NaN,NaN,NaN,271,NaN,NaN,NaN,NaN
1,Panther_lodging_Shelia,Panther,NaN,0.0,Lodging/residential,Residence Hall,929.0,10000.0,28.517689,-81.379039,...,1992.0,NaN,NaN,NaN,NaN,62,NaN,NaN,NaN,NaN
2,Panther_lodging_Ricky,Panther,NaN,0.0,Lodging/residential,Residence Hall,483.1,5200.0,28.517689,-81.379039,...,2016.0,NaN,NaN,NaN,NaN,534,NaN,NaN,NaN,NaN
3,Panther_education_Rosalie,Panther,0.0,0.0,Education,Research,690.5,7432.0,28.517689,-81.379039,...,2008.0,NaN,NaN,NaN,NaN,276,NaN,NaN,NaN,NaN
4,Panther_education_Misty,Panther,1.0,0.0,Education,Research,252.7,2720.0,28.517689,-81.379039,...,2004.0,NaN,NaN,NaN,NaN,375,NaN,NaN,NaN,NaN


In [4]:
print("--- Metadata Column Names and Non-Null Counts ---")
meta_df.info()


--- Metadata Column Names and Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 1636 entries, 0 to 1635
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   building_id            1636 non-null   str    
 1   site_id                1636 non-null   str    
 2   building_id_kaggle     1449 non-null   float64
 3   site_id_kaggle         1599 non-null   float64
 4   primaryspaceusage      1615 non-null   str    
 5   sub_primaryspaceusage  1615 non-null   str    
 6   sqm                    1636 non-null   float64
 7   sqft                   1636 non-null   float64
 8   lat                    1399 non-null   float64
 9   lng                    1399 non-null   float64
 10  timezone               1636 non-null   str    
 11  electricity            1578 non-null   str    
 12  hotwater               185 non-null    str    
 13  chilledwater           555 non-null    str    
 14  steam            

In [5]:
print("--- Primary Space Usage Distribution ---")
display(meta_df["primaryspaceusage"].value_counts(dropna=False).to_frame())

print("\n--- Presence of Electricity Meters Across Buildings ---")
display(meta_df["electricity"].value_counts(dropna=False).to_frame())


--- Primary Space Usage Distribution ---


,count
primaryspaceusage,
Education,617
Office,307
Entertainment/public assembly,204
Lodging/residential,168
Public services,166
Other,29
Healthcare,29
Parking,24
NaN,21



--- Presence of Electricity Meters Across Buildings ---


,count
electricity,
Yes,1578
NaN,58


> **Critical Finding on BDG2 Data Limitations:**  
> Notice that `occupants` has 1,406 missing values out of 1,636 buildings (85.94% missing), and `energystarscore` has 1,473 missing values (90.04% missing).  
> **Key Academic Insight:** BDG2 does NOT contain actual dynamic occupancy sensor streams, detailed operational logs, or appliance-level sub-metering. Any human occupancy or building activity patterns must be modeled through calendar and time proxies (Day of Week, Weekend Indicator, Month) rather than claimed as measured physical variables.


### 4. Inspecting Weather Data (`weather.csv`)
Let us inspect the hourly outdoor weather measurements across the 19 campus and city sites.


In [6]:
weather_df = load_weather(raw_data_dir, parse_dates=True)
print(f"Weather Data Dimensions: {weather_df.shape[0]} rows, {weather_df.shape[1]} columns")
print(f"Timestamp Date Range: {weather_df['timestamp'].min()} to {weather_df['timestamp'].max()}")
print(f"Unique Sites ({weather_df['site_id'].nunique()}): {weather_df['site_id'].unique().tolist()}")
display(weather_df.head(5))


Weather Data Dimensions: 331166 rows, 10 columns
Timestamp Date Range: 2016-01-01 00:00:00 to 2017-12-31 23:00:00
Unique Sites (19): ['Panther', 'Robin', 'Fox', 'Rat', 'Bear', 'Lamb', 'Peacock', 'Moose', 'Gator', 'Bull', 'Bobcat', 'Crow', 'Shrew', 'Swan', 'Wolf', 'Hog', 'Eagle', 'Cockatoo', 'Mouse']


,timestamp,site_id,airTemperature,cloudCoverage,dewTemperature,precipDepth1HR,precipDepth6HR,seaLvlPressure,windDirection,windSpeed
0,2016-01-01 00:00:00,Panther,19.4,NaN,19.4,0.0,NaN,NaN,0.0,0.0
1,2016-01-01 01:00:00,Panther,21.1,6.0,21.1,-1.0,NaN,1019.4,0.0,0.0
2,2016-01-01 02:00:00,Panther,21.1,NaN,21.1,0.0,NaN,1018.8,210.0,1.5
3,2016-01-01 03:00:00,Panther,20.6,NaN,20.0,0.0,NaN,1018.1,0.0,0.0
4,2016-01-01 04:00:00,Panther,21.1,NaN,20.6,0.0,NaN,1019.0,290.0,1.5


In [7]:
print("--- Weather Data Missingness Analysis ---")
weather_missing = pd.DataFrame({
    "Missing Count": weather_df.isnull().sum(),
    "Missing Percentage (%)": (weather_df.isnull().mean() * 100).round(2),
    "Data Type": weather_df.dtypes
})
display(weather_missing)


--- Weather Data Missingness Analysis ---


,Missing Count,Missing Percentage (%),Data Type
timestamp,0,0.00,datetime64[us]
site_id,0,0.00,str
airTemperature,128,0.04,float64
cloudCoverage,170987,51.63,float64
dewTemperature,328,0.10,float64
precipDepth1HR,133186,40.22,float64
precipDepth6HR,313004,94.52,float64
seaLvlPressure,21624,6.53,float64
windDirection,13005,3.93,float64
windSpeed,574,0.17,float64


In [8]:
print("--- Physical Weather Variables Summary Statistics ---")
display(weather_df.describe().round(2))


--- Physical Weather Variables Summary Statistics ---


,timestamp,airTemperature,cloudCoverage,dewTemperature,precipDepth1HR,precipDepth6HR,seaLvlPressure,windDirection,windSpeed
count,331166,331038.00,160179.00,330838.00,197980.00,18162.00,309542.00,318161.00,330592.00
mean,2016-12-31 04:12:21.326102,14.24,1.92,7.65,0.96,13.54,1016.06,184.39,3.57
min,2016-01-01 00:00:00,-28.90,0.00,-35.00,-1.00,-1.00,968.20,0.00,0.00
25%,2016-07-01 20:00:00,7.80,0.00,1.80,0.00,0.00,1011.60,90.00,2.10
50%,2016-12-30 21:00:00,14.40,0.00,8.50,0.00,0.00,1016.20,200.00,3.10
75%,2017-07-01 18:00:00,21.10,4.00,13.90,0.00,5.00,1020.90,280.00,5.00
max,2017-12-31 23:00:00,48.30,9.00,26.70,597.00,770.00,1050.10,360.00,24.20
std,NaN,9.99,2.55,9.20,8.27,43.80,8.05,111.57,2.34


> **Verified Physical Measurement Units:**
> - `airTemperature`: Ambient dry-bulb temperature in **degrees Celsius (°C)**. Mean = 14.24°C, Range = -28.9°C to 48.3°C. (99.96% complete)
> - `dewTemperature`: Dew point temperature in **degrees Celsius (°C)**. (99.90% complete)
> - `windSpeed`: Wind speed in **meters per second (m/s)**. (99.83% complete)
> - `seaLvlPressure`: Atmospheric pressure in **millibars / hPa**. (93.47% complete)
> - `cloudCoverage`: Cloud cover in **oktas** (0 to 9 scale). (48.37% complete, 51.63% missing)
> - `precipDepth1HR`: 1-hour liquid precipitation accumulation in **millimeters (mm)**. (40.22% missing)
> - `precipDepth6HR`: 6-hour liquid precipitation accumulation in **millimeters (mm)**. (94.52% missing)


### 5. Inspecting Electricity Meter Data (`electricity.csv` vs `electricity_cleaned.csv`)
Let us inspect the structure, frequency, and units of the hourly electricity meter readings.


In [9]:
raw_path = raw_data_dir / "meters" / "raw" / "electricity.csv"
clean_path = raw_data_dir / "meters" / "cleaned" / "electricity_cleaned.csv"

raw_header = pd.read_csv(raw_path, nrows=0)
clean_header = pd.read_csv(clean_path, nrows=0)

print(f"Raw Electricity Columns: {len(raw_header.columns)} (1 timestamp + {len(raw_header.columns)-1} buildings)")
print(f"Cleaned Electricity Columns: {len(clean_header.columns)} (1 timestamp + {len(clean_header.columns)-1} buildings)")
print(f"Do column names match exactly?: {list(raw_header.columns) == list(clean_header.columns)}")


Raw Electricity Columns: 1579 (1 timestamp + 1578 buildings)
Cleaned Electricity Columns: 1579 (1 timestamp + 1578 buildings)
Do column names match exactly?: True


In [10]:
# Select sample buildings that actually have electricity meters
sample_buildings = meta_df[meta_df["electricity"] == "Yes"]["building_id"].head(5).tolist()
print("Sample Building IDs:", sample_buildings)

elec_sample_raw = pd.read_csv(raw_path, usecols=["timestamp"] + sample_buildings, parse_dates=["timestamp"], index_col="timestamp")
elec_sample_clean = pd.read_csv(clean_path, usecols=["timestamp"] + sample_buildings, parse_dates=["timestamp"], index_col="timestamp")

print("Sample Raw Readings (Head):")
display(elec_sample_raw.head(3))
print("Sample Cleaned Readings (Head):")
display(elec_sample_clean.head(3))


Sample Building IDs: ['Panther_education_Rosalie', 'Panther_education_Misty', 'Panther_education_Mattie', 'Panther_education_Diann', 'Panther_education_Gina']
Sample Raw Readings (Head):


,Panther_education_Misty,Panther_education_Rosalie,Panther_education_Mattie,Panther_education_Diann,Panther_education_Gina
timestamp,,,,,
2016-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0
2016-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0
2016-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0


Sample Cleaned Readings (Head):


,Panther_education_Misty,Panther_education_Rosalie,Panther_education_Mattie,Panther_education_Diann,Panther_education_Gina
timestamp,,,,,
2016-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0
2016-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0
2016-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0


In [11]:
print("--- Raw vs Cleaned Comparison for 5 Sample Buildings ---")
comp_df = pd.DataFrame({
    "Raw NaNs": elec_sample_raw.isnull().sum(),
    "Cleaned NaNs": elec_sample_clean.isnull().sum(),
    "Raw Zeros": (elec_sample_raw == 0).sum(),
    "Cleaned Zeros": (elec_sample_clean == 0).sum(),
    "Raw Mean (kWh)": elec_sample_raw.mean().round(2),
    "Cleaned Mean (kWh)": elec_sample_clean.mean().round(2),
})
display(comp_df)


--- Raw vs Cleaned Comparison for 5 Sample Buildings ---


,Raw NaNs,Cleaned NaNs,Raw Zeros,Cleaned Zeros,Raw Mean (kWh),Cleaned Mean (kWh)
Panther_education_Misty,20,20,3237,3237,27.26,27.26
Panther_education_Rosalie,20,20,3373,3373,52.47,52.47
Panther_education_Mattie,12,12,3374,3374,5.40,5.40
Panther_education_Diann,12,12,3379,3379,72.34,72.34
Panther_education_Gina,11,12,3352,3352,337.39,337.35


> **Verified Electricity Measurement Unit and Frequency:**  
> - **Measurement Unit:** Hourly energy consumption in **kilowatt-hours (kWh)**. (Verified via BDG2 official repository documentation and Miller et al., 2020).
> - **Time Frequency:** Exactly 1-hour intervals (`00:00:00`, `01:00:00`, ..., `23:00:00`).
> - **Total Row Count:** 17,544 rows = 731 days × 24 hours (2016 leap year: 366 days + 2017: 365 days).
> - **Aggregation Implication for Phase 4:** Because readings are already **energy in kWh per hour** (not instantaneous power in kW), daily electricity consumption is obtained by taking the sum of valid hourly kWh readings over each 24-hour day ($\\text{Daily kWh} = \\sum_{h=1}^{24} \\text{Hourly kWh}_h$).


### 6. Cross-Dataset Linkage and Consistency
Let us verify the relational integrity across metadata, weather, and electricity meter datasets.


In [12]:
# Verify site consistency
meta_sites = set(meta_df["site_id"].unique())
weather_sites = set(weather_df["site_id"].unique())

print(f"Unique Sites in Metadata: {len(meta_sites)}")
print(f"Unique Sites in Weather: {len(weather_sites)}")
print(f"Are all metadata sites present in weather data?: {meta_sites == weather_sites}")


Unique Sites in Metadata: 19
Unique Sites in Weather: 19
Are all metadata sites present in weather data?: True


In [13]:
# Breakdown of buildings and timezones per site
site_summary = meta_df.groupby(["site_id", "timezone"]).agg(
    total_buildings=("building_id", "count"),
    buildings_with_electricity=("electricity", lambda x: (x == "Yes").sum())
).reset_index()
display(site_summary.sort_values(by="total_buildings", ascending=False))


,site_id,timezone,total_buildings,buildings_with_electricity
14,Rat,US/Eastern,305,305
8,Hog,US/Central,163,152
9,Lamb,Europe/London,147,146
6,Fox,US/Mountain,137,137
12,Panther,US/Eastern,136,105
2,Bull,US/Central,124,123
3,Cockatoo,US/Eastern,124,117
5,Eagle,US/Eastern,106,106
0,Bear,US/Pacific,92,92
7,Gator,US/Eastern,74,74


### 7. Phase 1 Key Findings Summary Table
| Dimension | Verified Parameter / Finding | Academic & Engineering Implication |
| :--- | :--- | :--- |
| **Dataset Title** | Building Data Genome Project 2 (BDG2) | Real-world non-residential building energy benchmark |
| **Target Variable** | Electricity Meter Reading | Target unit: **kilowatt-hours (kWh)** per hour; aggregated to **kWh/day** |
| **Time Span** | 2016-01-01 00:00:00 to 2017-12-31 23:00:00 | Exactly 2 full calendar years (731 days, 17,544 hourly timestamps) |
| **Building Universe** | 1,636 total buildings across 19 sites | Exactly 1,578 buildings possess an active electricity meter |
| **Primary Building Types** | Education (617), Office (307), Public Assembly (204) | Non-residential usage dominating commercial / institutional profile |
| **Weather Linkage** | 19 unique site weather stations | Ambient temperature (°C) and dew point (°C) have >99.9% completeness |
| **Missingness in Electricity** | Raw: 4.74% NaNs, 4.19% zeros; Cleaned: 5.43% NaNs | Anomaly spikes and persistent zero dropouts (>24h) converted to NaN |
| **Data Limitations** | Occupancy count is 85.9% missing; no appliance sensors | Occupancy patterns must be modeled via calendar proxies, not assumed |


In [14]:
print("Phase 1: Dataset Inspection and Understanding completed successfully!")
print("All checks, column types, shapes, and units verified on the extracted raw dataset.")


Phase 1: Dataset Inspection and Understanding completed successfully!
All checks, column types, shapes, and units verified on the extracted raw dataset.
